# Lunar engine — real-world cases

Four concrete scenarios a user hits in production, wired against the Go engine shipped inside the `lunar-router` wheel:

1. **Support-ticket classifier** — 20 real-shaped tickets, routed. Measure cost savings vs always-GPT-4o.
2. **Mixed workload** — Q&A + code review + creative writing. See which task types fall into which clusters, and how the router splits cost per category.
3. **Drop-in OpenAI gateway** — point the OpenAI SDK at the engine's local URL, make real chat-completions, and inspect the trace metadata the engine attaches.
4. **Observability** — what the engine recorded about the traffic you just generated.

Cases 1–2 run fully offline (routing decisions + pricing lookup). Cases 3–4 require `OPENAI_API_KEY` to do live calls — they gate themselves and skip cleanly otherwise.

In [1]:
import os, sys, time, json, statistics, urllib.request

SRC = os.path.abspath(os.path.join(os.getcwd(), ".."))
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import lunar_router as lr
print(f"lunar-router {lr.__version__}")

# Boot the engine. `load_router()` spawns the Go subprocess, bundled from the
# wheel — no system deps, no `make build`.
router = lr.load_router(cost_weight=0.5, verbose=True)
engine_url = router._engine._base_url
print(f"\nengine listening at: {engine_url}")
print(f"health:              {router._engine.health()}")

lunar-router 0.2.0
Package 'weights-default' is already installed at /home/ubuntu/.local/share/lunar_router/weights-default
Loading Go engine with weights: /home/ubuntu/.local/share/lunar_router/weights-default
Go engine ready! Models: 10, Clusters: 100, Embedder: True

engine listening at: http://127.0.0.1:56197
health:              {'status': 'healthy', 'router_initialized': True, 'num_models': 10, 'num_clusters': 100, 'embedder_ready': True}


## Case 1 — Support-ticket classifier

You run a SaaS. Twenty tickets a day need to be triaged into `billing | technical | feature_request`. Naïve answer: pipe every ticket through GPT-4o. That's ~600/month but it's overkill — half of the tickets are "where's my invoice?" which a cheap model handles fine.

Here's the same workload routed through Lunar.

In [2]:
# 20 realistically-shaped tickets (de-identified). Mix of easy billing
# queries, genuinely tricky technical bugs, and feature requests.
tickets = [
    "Where can I download my invoice for March? I don't see it in the dashboard.",
    "Tried to upgrade my plan and the checkout flashes an error: 'card declined' but my card is fine on other sites.",
    "Can you add support for exporting traces as parquet? Our analytics team wants column-oriented ingestion.",
    "My webhook endpoint stops receiving events after ~30min. Redelivery retries work. Looks like keepalive issue?",
    "What's the difference between the Starter and Growth plan?",
    "Forgot my password. Reset email isn't arriving; checked spam.",
    "Spike in 504 responses starting 14:22 UTC. Only affects our EU tenant. Gateway timing out upstream.",
    "Feature idea: per-route cost caps so a runaway loop doesn't bankrupt us overnight.",
    "Does billing prorate on mid-cycle plan changes?",
    "We're seeing stale results from the router when we swap a model alias. Is there a cache TTL we need to bust?",
    "Asked a question yesterday, no response. Ticket #8412.",
    "Can I use my own embedding model for clustering? We have a domain-tuned model that'd fit better than MiniLM.",
    "When I call /v1/chat/completions with tools=[...], tool_choice='required' is ignored on anthropic routes.",
    "Hey there — love the product! Quick q: annual billing available?",
    "Our ClickHouse ingestion is lagging by ~30 min during peak traffic. Flush interval config option?",
    "How do I cancel my subscription?",
    "Add a 'copy as curl' button on the trace detail page — super useful for reproducing errors.",
    "SSO login loops back to the login page after auth. Okta-backed, SAML. Browser network tab shows a redirect to /auth/callback then back to /login.",
    "Do you charge tax for Brazilian customers?",
    "Distillation job stuck at phase=curate for 2h+. Logs show no progress. Cancel and retry, or is something hung?",
]
print(f"workload: {len(tickets)} tickets")

workload: 20 tickets


In [3]:
# Route each ticket. Record selected model + cluster + expected error.
t0 = time.time()
decisions = [router.route(t) for t in tickets]
elapsed = time.time() - t0

from collections import Counter
models_picked = Counter(d.selected_model for d in decisions)
clusters_hit  = Counter(d.cluster_id for d in decisions)

print(f"routed {len(tickets)} tickets in {elapsed*1000:.0f}ms  ({1000*elapsed/len(tickets):.1f}ms/ticket)")
print(f"\nmodel distribution:")
for model, n in models_picked.most_common():
    bar = "█" * n
    print(f"  {model:<30} {bar} {n}")
print(f"\ndistinct clusters touched: {len(clusters_hit)} / 100")

routed 20 tickets in 179ms  (8.9ms/ticket)

model distribution:
  gpt-4o                         ██████████ 10
  gpt-3.5-turbo                  ███ 3
  ministral-3b-latest            ███ 3
  pixtral-12b-2409               ██ 2
  gpt-4o-mini                    █ 1
  mistral-small-latest           █ 1

distinct clusters touched: 14 / 100


In [4]:
# Cost comparison — routed vs always-GPT-4o.
# Assume avg 80 input tokens / 60 output tokens per ticket (realistic for
# triage-style classifications). Multiply by 20 tickets/day * 30 days.
IN_TOKS, OUT_TOKS = 80, 60
DAILY_VOLUME = 20
MONTHLY_VOLUME = DAILY_VOLUME * 30

def monthly_cost(model, n_per_month):
    return lr.model_cost(model, IN_TOKS, OUT_TOKS) * n_per_month

# Routed: sum over actual picks, scaled to monthly
picks_ratio = {m: n/len(tickets) for m, n in models_picked.items()}
routed_monthly = sum(monthly_cost(m, int(r * MONTHLY_VOLUME)) for m, r in picks_ratio.items())

# Baselines
always_gpt4o_monthly = monthly_cost("gpt-4o", MONTHLY_VOLUME)
always_mini_monthly  = monthly_cost("gpt-4o-mini", MONTHLY_VOLUME)

print(f"Projected monthly cost ({MONTHLY_VOLUME} tickets/month, ~{IN_TOKS}in/{OUT_TOKS}out):")
print(f"  always gpt-4o           ${always_gpt4o_monthly:>8.4f}  (baseline)")
print(f"  lunar auto-route        ${routed_monthly:>8.4f}  ({100*(1-routed_monthly/always_gpt4o_monthly):>5.1f}% savings vs gpt-4o)")
print(f"  always gpt-4o-mini      ${always_mini_monthly:>8.4f}  (floor — but quality risk on hard tickets)")

Projected monthly cost (600 tickets/month, ~80in/60out):
  always gpt-4o           $  0.4800  (baseline)
  lunar auto-route        $  0.2565  ( 46.6% savings vs gpt-4o)
  always gpt-4o-mini      $  0.0288  (floor — but quality risk on hard tickets)


## Case 2 — Mixed workload

Your app mixes three kinds of traffic:
- **Q&A** — short factual/reasoning questions
- **Code** — code review + generation
- **Creative** — marketing copy, slogans, short creative writing

Each category has a different optimal model. The router should split them across models instead of paying gpt-4o rates for everything.

In [5]:
workload = {
    "qa": [
        "What is the boiling point of water at sea level?",
        "Who discovered penicillin?",
        "Difference between TCP and UDP?",
        "How does photosynthesis work?",
        "What does the 'free' command show on Linux?",
        "Who wrote Don Quixote?",
        "What year was Python first released?",
        "Is Pluto a planet?",
        "What causes the aurora borealis?",
        "Largest country by area?",
    ],
    "code": [
        "Write a Python decorator that caches function results with a TTL.",
        "Review this: `def foo(x=[]): x.append(1); return x` — bug and fix?",
        "Implement a thread-safe LRU cache in Rust using a Mutex.",
        "Why is `i++` not atomic? How to fix in Go?",
        "Write a SQL query: top 5 products by revenue in Q1 2025, grouped by category.",
        "Explain the difference between Promise.all and Promise.allSettled.",
        "Convert this recursive Fibonacci to iterative with O(1) space.",
        "Detect deadlocks in this Go code snippet: [channel swap scenario]",
        "Write a Dockerfile for a Python FastAPI app with multi-stage build.",
        "Refactor this 40-line callback pyramid into async/await.",
    ],
    "creative": [
        "Write a haiku about a lonely lighthouse.",
        "Generate 5 slogan candidates for an eco-friendly coffee brand.",
        "Draft a 2-sentence LinkedIn post about shipping a v1.",
        "Come up with a name for a minimalist task-tracker app.",
        "Describe the smell of autumn in two evocative sentences.",
        "Write an opening line for a sci-fi novel about memory trading.",
        "Rewrite 'we launched a thing' as an exciting announcement tweet.",
        "Pitch a children's book about a shy octopus in 3 lines.",
        "Three taglines for a fintech app that helps freelancers save.",
        "Write a short personal note thanking a teacher who influenced you.",
    ],
}
print(f"workload breakdown: {({k: len(v) for k,v in workload.items()})}")

workload breakdown: {'qa': 10, 'code': 10, 'creative': 10}


In [6]:
# Route each, group decisions by task category.
from collections import defaultdict

picks_by_cat = defaultdict(Counter)
errs_by_cat  = defaultdict(list)
all_picks    = Counter()

for cat, prompts in workload.items():
    for p in prompts:
        d = router.route(p)
        picks_by_cat[cat][d.selected_model] += 1
        errs_by_cat[cat].append(d.expected_error)
        all_picks[d.selected_model] += 1

print(f"{'category':<10} {'picked models (top 3)':<55} {'avg err':>8}")
print("-" * 75)
for cat in workload:
    top = ", ".join(f"{m}×{n}" for m, n in picks_by_cat[cat].most_common(3))
    avg_err = statistics.mean(errs_by_cat[cat])
    print(f"{cat:<10} {top:<55} {avg_err:>8.3f}")

print(f"\naggregate picks across all 30 prompts:")
for m, n in all_picks.most_common():
    print(f"  {m:<30} {n}")

category   picked models (top 3)                                    avg err
---------------------------------------------------------------------------
qa         mistral-small-latest×3, ministral-3b-latest×3, gpt-4o×2    0.087
code       gpt-4o×7, ministral-8b-latest×1, gpt-3.5-turbo×1           0.086
creative   gpt-4o×6, ministral-3b-latest×4                            0.161

aggregate picks across all 30 prompts:
  gpt-4o                         15
  ministral-3b-latest            8
  mistral-small-latest           3
  pixtral-12b-2409               1
  gpt-4o-mini                    1
  ministral-8b-latest            1
  gpt-3.5-turbo                  1


In [7]:
# Project to a real monthly cost. Say the mix is 60% Q&A, 25% code, 15% creative,
# and total volume is 10k requests/month, avg 120in/180out tokens.
MIX       = {"qa": 0.60, "code": 0.25, "creative": 0.15}
VOLUME    = 10_000
IN_T, OUT_T = 120, 180

def projected_monthly_for_category(cat):
    picks = picks_by_cat[cat]
    total_routed = sum(picks.values())
    volume_cat = VOLUME * MIX[cat]
    return sum(
        lr.model_cost(m, IN_T, OUT_T) * volume_cat * (n / total_routed)
        for m, n in picks.items()
    )

routed_total        = sum(projected_monthly_for_category(c) for c in workload)
always_gpt4o_total  = lr.model_cost("gpt-4o", IN_T, OUT_T) * VOLUME
always_mini_total   = lr.model_cost("gpt-4o-mini", IN_T, OUT_T) * VOLUME

print(f"Projected monthly cost  (volume={VOLUME:,}, mix={MIX})")
print(f"  always gpt-4o       ${always_gpt4o_total:>9.2f}   (baseline)")
print(f"  lunar auto-route    ${routed_total:>9.2f}   ({100*(1-routed_total/always_gpt4o_total):>5.1f}% savings)")
print(f"  always gpt-4o-mini  ${always_mini_total:>9.2f}   (cheapest, ignores quality)")

Projected monthly cost  (volume=10,000, mix={'qa': 0.6, 'code': 0.25, 'creative': 0.15})
  always gpt-4o       $    21.00   (baseline)
  lunar auto-route    $     8.55   ( 59.3% savings)
  always gpt-4o-mini  $     1.26   (cheapest, ignores quality)


## Case 3 — Drop-in OpenAI gateway

The engine is already a `/v1/chat/completions` server. Point the OpenAI SDK at it and it works — same API, same response shape. The engine adds provider fan-out, fallback, and cost/latency tracking.

Gated on `OPENAI_API_KEY` so the notebook stays runnable without secrets. Set the env var before running to see real traffic.

In [8]:
# Show the engine URL and what it exposes
print(f"engine url:    {engine_url}")
print(f"try it raw:    curl {engine_url}/health")
try:
    with urllib.request.urlopen(f"{engine_url}/health", timeout=2) as r:
        print(f"health:        {json.loads(r.read())}")
except Exception as e:
    print(f"health probe failed: {e}")

engine url:    http://127.0.0.1:56197
try it raw:    curl http://127.0.0.1:56197/health
health:        {'status': 'healthy', 'router_initialized': True, 'num_models': 10, 'num_clusters': 100, 'embedder_ready': True}


In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    print("set OPENAI_API_KEY to run this cell; skipping.")
else:
    # lr.completion is the thin shim — point it at the engine, each request
    # becomes a trace. `_cost`, `_latency_ms`, `_routing` ride back on the response.
    prompts = [
        ("easy", "What is 2+2? Reply with just the number."),
        ("med",  "Write one sentence describing how TLS handshakes work."),
        ("hard", "List 3 edge cases to test in a distributed rate-limiter."),
    ]
    total_cost = 0.0
    total_latency = 0.0
    for label, p in prompts:
        resp = lr.completion(
            model="openai/gpt-4o-mini",
            messages=[{"role": "user", "content": p}],
            max_tokens=60, temperature=0,
            api_base=f"{engine_url}/v1",
            api_key=os.environ["OPENAI_API_KEY"],
        )
        answer = resp.choices[0].message.content.strip().replace("\n", " ")[:80]
        cost = resp._cost or 0.0
        latency = resp._latency_ms or 0.0
        total_cost += cost
        total_latency += latency
        print(f"[{label:<4}] ${cost:.6f}  {latency:>5.0f}ms  →  {answer}")
    print(f"\n{'':>7} ${total_cost:.6f}  {total_latency:>5.0f}ms total across {len(prompts)} requests")

## Case 4 — Observability

The engine exposes introspection endpoints. This cell queries what the running engine knows about itself: which models are loaded, how many clusters, and (when the API server is running) trace/cost summaries.

In [ ]:
def _get(path, timeout=2):
    try:
        with urllib.request.urlopen(f"{engine_url}{path}", timeout=timeout) as r:
            return r.status, json.loads(r.read())
    except Exception as e:
        return None, str(e)

for path in ["/health", "/v1/models"]:
    code, body = _get(path)
    body_str = json.dumps(body, default=str)[:120] if code else body
    print(f"  GET {path:<20} → {code}  {body_str}")

# If the REST API server is also up (make start-full), query higher-level analytics.
import urllib.error
API = "http://localhost:8000"
try:
    with urllib.request.urlopen(f"{API}/health", timeout=1) as r:
        print(f"\n  API server up at {API}")
        with urllib.request.urlopen(f"{API}/v1/traces?limit=5", timeout=2) as r:
            traces = json.loads(r.read())
            print(f"  recent traces (top 5 of {traces.get('total', '?')}):")
            for t in traces.get("items", [])[:5]:
                print(f"    {t.get('timestamp','')[:19]}  {t.get('model','?'):<30}  ${t.get('cost',0):.5f}")
except Exception:
    print(f"\n  API server at {API} not reachable (run `make start-full` to enable trace analytics)")

## Wrap-up

Three takeaways from the data above:

1. **Routing pays off where workload is heterogeneous.** On homogeneous batches (all-hard, all-easy) the router converges to the same model and savings disappear. The wedge is real for mixed workloads.
2. **Routing decisions are ~ms, so the latency cost is effectively zero** compared to the LLM call itself (hundreds of ms).
3. **Drop-in gateway mode means no code changes.** Change `base_url` once; every request becomes a trace you can later turn into a distilled student model.

Cleanup (kill the spawned engine subprocess):

In [ ]:
router._engine.stop()
print("engine stopped")